In [1]:
import math

class Node:
    def __init__(self, value=None, name=""):
        self.value = value
        self.name = name
        self.children = []
        self.minmax_value = None

class Environment:
    def __init__(self):
        self.states = []

    def compute_minimax(self, node, depth, maximizing):
        self.states.append(node.name or str(node.value))
        if not node.children:
            return node.value
        if maximizing:
            best = -math.inf
            for child in node.children:
                val = self.compute_minimax(child, depth-1, False)
                best = max(best, val)
            node.minmax_value = best
            return best
        else:
            best = math.inf
            for child in node.children:
                val = self.compute_minimax(child, depth-1, True)
                best = min(best, val)
            node.minmax_value = best
            return best

class MinimaxAgent:
    def act(self, root, env):
        result = env.compute_minimax(root, 2, True)
        root.minmax_value = result

# Build tree
root  = Node(name="Warehouse")
zA    = Node(name="Zone A")
zB    = Node(name="Zone B")
root.children = [zA, zB]
zA.children = [Node(3, "ZA-Opt1"), Node(5, "ZA-Opt2")]
zB.children = [Node(2, "ZB-Opt1"), Node(9, "ZB-Opt2")]

env   = Environment()
agent = MinimaxAgent()
agent.act(root, env)

print("States visited:", env.states)
print("Zone A value:", zA.minmax_value)
print("Zone B value:", zB.minmax_value)
print("Best move for Robot:", "Zone A" if zA.minmax_value > zB.minmax_value else "Zone B")
print("Optimal safety value:", root.minmax_value)

States visited: ['Warehouse', 'Zone A', 'ZA-Opt1', 'ZA-Opt2', 'Zone B', 'ZB-Opt1', 'ZB-Opt2']
Zone A value: 3
Zone B value: 2
Best move for Robot: Zone A
Optimal safety value: 3


In [2]:
import math

class Node:
    def __init__(self, value=None, name=""):
        self.value = value
        self.name = name
        self.children = []

def minimax(node, maximizing):
    if not node.children:
        return node.value
    if maximizing:
        return max(minimax(c, False) for c in node.children)
    else:
        return min(minimax(c, True) for c in node.children)

# Build tree
root = Node(name="Start")
stop = Node(name="Stop")
go   = Node(name="Go")
turn = Node(name="Turn")
root.children = [stop, go, turn]

stop.children = [Node(6), Node(7)]
go.children   = [Node(2), Node(4)]
turn.children = [Node(8), Node(3)]

for child in root.children:
    child.value = minimax(child, False)  # Min's turn next

print("Stop:", stop.value)
print("Go:  ", go.value)
print("Turn:", turn.value)
print("Best action:", max(root.children, key=lambda n: n.value).name)
print("Optimal value:", max(c.value for c in root.children))

Stop: 6
Go:   2
Turn: 3
Best action: Stop
Optimal value: 6


In [8]:
import math

class Node:
    def __init__(self, value=None, name=""):
        self.value = value
        self.name = name
        self.children = []
        self.minmax_value = None

def alpha_beta(node, alpha, beta, maximizing):
    if not node.children:
        return node.value

    if maximizing:
        best = -math.inf
        for child in node.children:
            val = alpha_beta(child, alpha, beta, False)
            best = max(best, val)
            alpha = max(alpha, best)
            if beta <= alpha:
                print(f"    β({beta}) ≤ α({alpha}) → PRUNE {child.name or child.value} ✂️")
                break
        node.minmax_value = best
        return best
    else:
        best = math.inf
        for child in node.children:
            print(f"    → leaf {child.value} → β=min({beta},{child.value})={min(beta,child.value)}")
            val = alpha_beta(child, alpha, beta, True)
            best = min(best, val)
            beta = min(beta, best)
            if beta <= alpha:
                print(f"    β({beta}) ≤ α({alpha}) → PRUNE next leaf")
                break
        node.minmax_value = best
        return best

# Build tree
root = Node(name="Start")
stop = Node(name="Stop")
go   = Node(name="Go")
turn = Node(name="Turn")
root.children = [stop, go, turn]

stop.children = [Node(6), Node(7)]
go.children   = [Node(2), Node(4)]
turn.children = [Node(8), Node(3)]

alpha = -math.inf
beta  = math.inf
print(f"Start (MAX), α=-∞, β=+∞\n")

for child in root.children:
    print(f"→ {child.name} (MIN), α={alpha if alpha != -math.inf else '-∞'}, β={beta if beta != math.inf else '+∞'}")
    val = alpha_beta(child, alpha, beta, False)
    alpha = max(alpha, val)
    print(f"  {child.name}={val}")
    print(f"  α=max({'-∞' if alpha == val else alpha-val+alpha},{val})={alpha}\n")

print(f"Final: Best = Stop (value={alpha})")

print("\n--- c) Pruned Branches ---")
print("Leaf 4 under Go is pruned (1 branch pruned)")

print("\n--- d) Nodes Evaluated ---")
print(f"Minimax    : 7 nodes (all evaluated)")
print(f"Alpha-Beta : 6 nodes (leaf 4 skipped)")
print(f"Nodes saved: 1 node not evaluated due to pruning")


Start (MAX), α=-∞, β=+∞

→ Stop (MIN), α=-∞, β=+∞
    → leaf 6 → β=min(inf,6)=6
    → leaf 7 → β=min(6,7)=6
  Stop=6
  α=max(-∞,6)=6

→ Go (MIN), α=6, β=+∞
    → leaf 2 → β=min(inf,2)=2
    β(2) ≤ α(6) → PRUNE next leaf
  Go=2
  α=max(10,2)=6

→ Turn (MIN), α=6, β=+∞
    → leaf 8 → β=min(inf,8)=8
    → leaf 3 → β=min(8,3)=3
    β(3) ≤ α(6) → PRUNE next leaf
  Turn=3
  α=max(9,3)=6

Final: Best = Stop (value=6)

--- c) Pruned Branches ---
Leaf 4 under Go is pruned (1 branch pruned)

--- d) Nodes Evaluated ---
Minimax    : 7 nodes (all evaluated)
Alpha-Beta : 6 nodes (leaf 4 skipped)
Nodes saved: 1 node not evaluated due to pruning


In [7]:
# a) Minimax Calculation:
#
#    MAX level (A1,A2,D1,D2,G1,G2):
#       A1 = max(5,6) = 6
#       A2 = max(7,4) = 7
#       D1 = max(3,8) = 8
#       D2 = max(6,2) = 6
#       G1 = max(1,9) = 9
#       G2 = max(4,7) = 7
#
#    MIN level (Attack, Defend, Gather):
#       Attack = min(6,7) = 6
#       Defend = min(8,6) = 6
#       Gather = min(9,7) = 7
#
#    MAX level (Root):
#       Root = max(6,6,7) = 7  --> Best Move: GATHER
#
#
# b) Alpha-Beta Pruning :
#
#    Root (MAX)  a=-inf  b=+inf
#
#    -> Attack (MIN)  a=-inf  b=+inf
#       -> A1 (MAX)  a=-inf  b=+inf
#          leaf 5 -> a=max(-inf,5)=5
#          leaf 6 -> a=max(5,6)=6   A1=6
#       b=min(+inf,6)=6
#
#       -> A2 (MAX)  a=-inf  b=6
#          leaf 7 -> a=max(-inf,7)=7
#          b(6) <= a(7) --> PRUNE leaf 4
#          A2=7
#       b=min(6,7)=6  Attack=6
#    a=max(-inf,6)=6
#
#    -> Defend (MIN)  a=6  b=+inf
#       -> D1 (MAX)  a=6  b=+inf
#          leaf 3 -> a stays 6  (3 < 6)
#          leaf 8 -> a=max(6,8)=8   D1=8
#       b=min(+inf,8)=8
#
#       -> D2 (MAX)  a=6  b=8
#          leaf 6 -> a stays 6
#          leaf 2 -> a stays 6   D2=6
#       b=min(8,6)=6
#       b(6) <= a(6) --> Defend=6
#    a=max(6,6)=6
#
#    -> Gather (MIN)  a=6  b=+inf
#       -> G1 (MAX)  a=6  b=+inf
#          leaf 1 -> a stays 6  (1 < 6)
#          leaf 9 -> a=max(6,9)=9   G1=9
#       b=min(+inf,9)=9
#
#       -> G2 (MAX)  a=6  b=9
#          leaf 4 -> a stays 6  (4 < 6)
#          leaf 7 -> a=max(6,7)=7   G2=7
#       b=min(9,7)=7
#       Gather=7
#    a=max(6,7)=7
#
#    Root = 7  --> Best Move: GATHER
#
#
# c) Pruned Branches:
#
#    leaf 4 under A2 is pruned
#    only 1 branch pruned in this tree
#
#
# d) Nodes Evaluated:
#
#    Minimax    --> 13 nodes (all evaluated)
#    Alpha-Beta --> 12 nodes (leaf 4 under A2 skipped)
#    Nodes saved: 1
#
#
# e) How Alpha-Beta Improves Efficiency:
#
#    once MAX finds a value >= beta (MIN's best),
#    there is no point checking further children
#    because MIN will never pick that branch anyway
#    so we skip it without affecting the final answer
#    in deeper trees like chess this saves huge computation